In [6]:
pip install wrds

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ACC102 Track 2
# Industry: Technology Giants (Apple, Microsoft, Google)
# Data Source: WRDS CRSP Database

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy.stats import linregress
import wrds

# ----------------------
# 1. Connect to WRDS
# ----------------------
db = wrds.Connection()
print("Connected to WRDS successfully.")

# ----------------------
# 2. Define 3 Tech Giants 
# ----------------------
firm_info = {
    "Apple": 14593,
    "Microsoft": 10107,
    "Google": 11308
}
start_dt = "2024-01-01"
end_dt = "2024-12-31"

permno_list = tuple(firm_info.values())

# WRDS SQL query
sql = f"""
SELECT permno, date, prc, ret, vol
FROM crsp.dsf
WHERE permno IN {permno_list}
  AND date >= '{start_dt}'
  AND date <= '{end_dt}'
"""

raw_data = db.raw_sql(sql, date_cols=["date"])
db.close()

# ----------------------
# 3. Data Cleaning & Preparation
# ----------------------
permno2name = {v:k for k,v in firm_info.items()}
raw_data["firm"] = raw_data["permno"].map(permno2name)
raw_data["prc"] = raw_data["prc"].abs()  # Fix negative prices in CRSP

# Create main dataframes
price_df = raw_data.pivot(index="date", columns="firm", values="prc")
ret_df = raw_data.pivot(index="date", columns="firm", values="ret")
vol_df = raw_data.pivot(index="date", columns="firm", values="vol")

# Clean data
price_df = price_df.dropna()
ret_df = ret_df.dropna()
vol_df = vol_df.dropna()

print("===== Data Overview =====")
print(f"Time Range: {price_df.index.min()} ~ {price_df.index.max()}")
print(f"Total Trading Days: {len(price_df)}")
print("Available firms:", price_df.columns.tolist())

# ----------------------
# 4. Enhanced Financial Metrics Calculation
# ----------------------
# 1. Basic price & return metrics
norm_price = price_df / price_df.iloc[0] * 100
cum_ret = (1 + ret_df).cumprod() - 1
annual_ret = ret_df.mean() * 252
annual_vol = ret_df.std() * np.sqrt(252)

# 2. Risk metrics
def max_dd(series):
    roll_max = series.cummax()
    drawdown = (series - roll_max) / roll_max
    return drawdown.min()

max_ddowns = pd.Series({c: max_dd(price_df[c]) for c in price_df.columns})

# 3. Risk-adjusted return metrics
risk_free = 0.02
daily_rf = risk_free / 252
sharpe_ratio = (ret_df.mean() - daily_rf) / ret_df.std() * np.sqrt(252)

# Sortino Ratio (uses only downside volatility)
downside_ret = ret_df.where(ret_df < daily_rf, np.nan)
downside_vol = downside_ret.std() * np.sqrt(252)
sortino_ratio = (ret_df.mean() * 252 - risk_free) / downside_vol

# 4. Regression-based metrics (vs Apple as benchmark)
benchmark_ret = ret_df["Apple"]
beta, alpha, r_squared, p_value, se = {}, {}, {}, {}, {}
for firm in ret_df.columns:
    if firm != "Apple":
        slope, intercept, r, p, std_err = linregress(benchmark_ret, ret_df[firm])
        beta[firm] = slope
        alpha[firm] = intercept * 252  # Annualized alpha
        r_squared[firm] = r**2
        p_value[firm] = p
        se[firm] = std_err

# 5. Volume metrics
avg_daily_vol = vol_df.mean()

# Create enhanced metrics table
metrics = pd.DataFrame({
    "Cumulative Return (%)": cum_ret.iloc[-1] * 100,
    "Annualized Return (%)": annual_ret * 100,
    "Annualized Volatility (%)": annual_vol * 100,
    "Max Drawdown (%)": max_ddowns * 100,
    "Sharpe Ratio": sharpe_ratio,
    "Sortino Ratio": sortino_ratio,
    "Avg Daily Volume": avg_daily_vol.astype(int)
})

# Add regression metrics to the table
regression_metrics = pd.DataFrame({
    "Beta (vs Apple)": pd.Series(beta),
    "Alpha (%)": pd.Series(alpha) * 100,
    "R-squared": pd.Series(r_squared),
    "P-value": pd.Series(p_value)
})

print("\n===== Enhanced Key Metrics =====")
print(metrics.round(2))
print("\n===== Regression Metrics (vs Apple) =====")
print(regression_metrics.round(3))

# ----------------------
# 5. Visualization (6 charts, removed problematic heatmap)
# ----------------------
plt.rcParams["figure.dpi"] = 300
sns.set_style("whitegrid")

# --- Chart 1: Normalized Price Trend ---
plt.figure(figsize=(14, 6))
norm_price.plot(linewidth=2.2)
plt.title("Normalized Stock Price Trend (Base = 100)\nApple vs Microsoft vs Google (2024)", fontsize=14)
plt.ylabel("Normalized Price")
plt.xlabel("Date")
plt.legend()
plt.tight_layout()
plt.savefig("1_normalized_price.png")
plt.close()

# --- Chart 2: Cumulative Return Comparison ---
plt.figure(figsize=(14, 6))
(cum_ret * 100).plot(linewidth=2.2)
plt.title("Cumulative Return Comparison (2024)", fontsize=14)
plt.ylabel("Cumulative Return (%)")
plt.xlabel("Date")
plt.legend()
plt.tight_layout()
plt.savefig("2_cumulative_return.png")
plt.close()

# --- Chart 3: Risk-Return Scatter Plot ---
plt.figure(figsize=(10, 6))
for i, firm in enumerate(metrics.index):
    plt.scatter(metrics["Annualized Volatility (%)"].iloc[i], metrics["Annualized Return (%)"].iloc[i], s=150, alpha=0.7)
    plt.annotate(firm, (metrics["Annualized Volatility (%)"].iloc[i], metrics["Annualized Return (%)"].iloc[i]), fontsize=12)
plt.title("Risk-Return Profile (2024)", fontsize=14)
plt.xlabel("Annualized Volatility (%)")
plt.ylabel("Annualized Return (%)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("3_risk_return_scatter.png")
plt.close()

# --- Chart 4: Rolling 30-Day Volatility ---
rolling_vol = ret_df.rolling(window=30).std() * np.sqrt(252) * 100

plt.figure(figsize=(14, 6))
rolling_vol.plot(linewidth=1.8)
plt.title("Rolling 30-Day Annualized Volatility (2024)", fontsize=14)
plt.ylabel("Volatility (%)")
plt.xlabel("Date")
plt.legend()
plt.tight_layout()
plt.savefig("4_rolling_volatility.png")
plt.close()

# --- Chart 5: Correlation Heatmap ---
corr_matrix = ret_df.corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Daily Return Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.savefig("5_correlation_heatmap.png")
plt.close()

# --- Chart 6: Daily Return Distribution ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for idx, col in enumerate(ret_df.columns):
    sns.histplot(ret_df[col], kde=True, ax=axes[idx], bins=40)
    axes[idx].set_title(f"{col} Daily Return Distribution")
plt.tight_layout()
plt.savefig("6_return_distribution.png")
plt.close()

# ----------------------
# 6. Enhanced Regression Analysis
# ----------------------
print("\n===== Enhanced Regression Analysis (Benchmark: Apple) =====")
for firm in ret_df.columns:
    if firm != "Apple":
        print(f"\n{firm} vs Apple:")
        print(f"  Beta Coefficient: {beta[firm]:.3f}")
        print(f"  Annualized Alpha: {alpha[firm]:.2f}%")
        print(f"  R-squared: {r_squared[firm]:.3f}")
        print(f"  P-value: {p_value[firm]:.4f}")

print("\n✅ Enhanced analysis completed! 6 charts and 2 metrics tables saved successfully.")